### Analysis Cell

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")

### Analysis Cell

In [ ]:
df_train = pd.read_csv("train.csv") 
df_test = pd.read_csv("test.csv")
print(f"Training set size: {df_train.shape}")
print(f"Testing set size: {df_test.shape}")
print(df_train.head())

### Analysis Cell

In [ ]:
print("Missing values check:")
print(df_train.isnull().sum())
plt.figure(figsize=(8, 5))
sns.countplot(x="Transported", data=df_train, palette="Set2")
plt.title("Distribution of Target Class")
plt.show()

### Analysis Cell

In [ ]:
numerical_features = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for feature in numerical_features:
    mean_val = df_train[feature].mean()
    df_train[feature] = df_train[feature].fillna(mean_val)
    df_test[feature] = df_test[feature].fillna(mean_val)
categorical_features = ['HomePlanet','CryoSleep','Cabin','Destination','VIP']
for feature in categorical_features:
    mode_val = df_train[feature].mode()[0]
    df_train[feature] = df_train[feature].fillna(mode_val)
    df_test[feature] = df_test[feature].fillna(mode_val)
df_train = pd.get_dummies(df_train, columns=categorical_features, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_features, drop_first=True)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

### Analysis Cell

In [ ]:
X_data = df_train.drop(['Transported','PassengerId','Name'], axis=1)
y_labels = df_train['Transported'].astype(int)
train_X, val_X, train_y, val_y = train_test_split(X_data, y_labels, test_size=0.2, random_state=42)

### Analysis Cell

In [ ]:
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(train_X, train_y)

### Analysis Cell

In [ ]:
predicted_y = classifier.predict(val_X)
score = accuracy_score(val_y, predicted_y)
print(f"Model Validation Accuracy: {score:.4f}")

### Analysis Cell

In [ ]:
final_X_test = df_test.drop(['PassengerId', 'Name'], axis=1, errors='ignore')
final_X_test = final_X_test.reindex(columns=train_X.columns, fill_value=0)
final_predictions = classifier.predict(final_X_test)
output_df = pd.DataFrame({
    "PassengerId": df_test['PassengerId'],
    "Transported": final_predictions.astype(bool)
})
output_df.to_csv("submission_final.csv", index=False)
print("Final submission file saved successfully.")